In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import shap
from src.scoutai_common import (
    get_engine, load_master_view, engineer_features, get_model,
    FULL_FEATURES, PERFORMANCE_FEATURES,
)

NAME_MAP = {
    'remainder__age': 'Player Age', 'remainder__max_career_transfer_fee': 'Max Transfer Fee',
    'remainder__most_recent_transfer_fee': 'Most Recent Transfer Fee',
    'remainder__international_caps': 'International Caps',
    'remainder__contract_months_remaining': 'Contract Remaining',
    'remainder__total_appearances': 'Total Appearances', 'remainder__stadium_seats': 'Stadium Capacity',
    'remainder__minutes_per_match': 'Minutes Per Match', 'remainder__foreigners_percentage': 'Foreigner Ratio',
    'remainder__has_transfer_history': 'Transfer History', 'remainder__club_avg_age': 'Club Avg Age',
    'remainder__total_goals': 'Total Goals', 'remainder__goals_per_90': 'Goals Per 90',
    'remainder__international_goals': 'International Goals', 'remainder__club_squad_size': 'Squad Size',
    'remainder__total_assists': 'Total Assists', 'remainder__height_in_cm': 'Height (cm)',
    'remainder__wonderkid_hype': 'Wonderkid Hype Index', 'remainder__assists_per_90': 'Assists Per 90',
    'remainder__total_yellow_cards': 'Yellow Cards', 'remainder__total_red_cards': 'Red Cards',
    'remainder__league_coefficient': 'League Quality', 'cat__passport_tier_Tier_1': 'Passport (Tier 1)',
    'cat__detailed_position_Goalkeeper': 'Position: Goalkeeper',
}

def clean_names(feature_names):
    return [NAME_MAP.get(n, n.replace('remainder__', '').replace('cat__', '').replace('_', ' ').title())
            for n in feature_names]

def run_shap_analysis(model_label, feature_list, subset_df):
    print(f"[SYSTEM] [{model_label}] Computing SHAP values ({len(subset_df)} players)...")
    pipeline = get_model(model_label)
    preprocessor = pipeline.named_steps['preprocessor']
    regressor = pipeline.named_steps['regressor']

    X_transformed = preprocessor.transform(subset_df[feature_list])
    names = clean_names(preprocessor.get_feature_names_out())

    explainer = shap.TreeExplainer(regressor)
    shap_values = explainer(X_transformed)

    plt.figure(figsize=(10, 10))
    shap.summary_plot(shap_values, X_transformed, feature_names=names, show=False)
    plt.title(f"ScoutAI ({model_label}): Global Feature Importance", fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("SHAP Value (Impact on predicted market value)", fontsize=12)
    plt.tight_layout()
    plt.savefig(f'../images/shap_summary_{model_label}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
engine = get_engine()
df = load_master_view(engine)
df = engineer_features(df)

df_full = df[df['has_transfer_history'] == 1].copy()
df_perf = df[df['has_transfer_history'] == 0].copy()

run_shap_analysis('full', FULL_FEATURES, df_full)
run_shap_analysis('performance_only', PERFORMANCE_FEATURES, df_perf)